In [1]:
import os
import nibabel as nib
import numpy as np
import h5py

base_dir = current_directory = os.getcwd()
base_dir = os.path.join(base_dir, "Data/train_v4/training")

print(base_dir)

Case_image_list = []
Case_mask_list = []
Case_name_list = []
for item in sorted(os.listdir(base_dir)):
    image_dir = os.path.join(base_dir, item, "preprocessed")
    mask_dir = os.path.join(base_dir, item, "masks")

    ## images
    if os.path.isdir(image_dir):
        images_loc = []
        for im in sorted(os.listdir(image_dir)):
            if im.endswith("t2_pp.nii"):
                im_array = nib.load(os.path.join(image_dir, im)).get_fdata()
                # print(f"case {im} image shape: {im_array.shape}")
                images_loc.append(im_array)
        images_loc = np.array(images_loc)
        Case_image_list.append(images_loc)
    else:
        print(f"no image directory found for {item}")


    ## masks
    if os.path.isdir(mask_dir):
        msk_loc = []
        for msk in sorted(os.listdir(mask_dir)):
            if msk.endswith("mask1.nii"):
                msk_array = nib.load(os.path.join(mask_dir, msk)).get_fdata()
                # print(f"case {msk} mask shape: {msk_array.shape}")
                # print(np.unique(msk_array))
                msk_loc.append(msk_array)
        msk_loc = np.array(msk_loc)
        print(f"case {item} mask shape: {msk_loc.shape}")
        Case_mask_list.append(msk_loc)
    else:
        print(f"no mask directory found for {item}")


    ## names, subtracted from the mask name
    if os.path.isdir(mask_dir):
        names_local = []
        for msk in sorted(os.listdir(mask_dir)):
            if msk.endswith("mask1.nii"):
                msk_name = msk[:-10]
                print(f"case {msk} mask name: {msk_name}")
                names_local.append(msk_name)
        Case_name_list.append(names_local)
    else:
        print(f"no mask directory found for {item}")



/home/jincan/longitudinal_segmentation/Data/train_v4/training
case training01 mask shape: (4, 181, 217, 181)
case training01_01_mask1.nii mask name: training01_01
case training01_02_mask1.nii mask name: training01_02
case training01_03_mask1.nii mask name: training01_03
case training01_04_mask1.nii mask name: training01_04
case training02 mask shape: (4, 181, 217, 181)
case training02_01_mask1.nii mask name: training02_01
case training02_02_mask1.nii mask name: training02_02
case training02_03_mask1.nii mask name: training02_03
case training02_04_mask1.nii mask name: training02_04
case training03 mask shape: (5, 181, 217, 181)
case training03_01_mask1.nii mask name: training03_01
case training03_02_mask1.nii mask name: training03_02
case training03_03_mask1.nii mask name: training03_03
case training03_04_mask1.nii mask name: training03_04
case training03_05_mask1.nii mask name: training03_05
case training04 mask shape: (4, 181, 217, 181)
case training04_01_mask1.nii mask name: training

In [2]:


### 4 files for training $ 1 for validation
### This is for pretraining dataset

## create training .hdf5 file 
if not os.path.exists('h5_Data'):
    os.mkdir("h5_Data/")
with h5py.File('h5_Data/train_long.hdf5', 'w') as f:
    for i in range(len(Case_name_list)-1):
        time_points = len(Case_name_list[i])
        grp_name = 'subject_' + str(i).zfill(3)
        print(grp_name)
        grp = f.create_group(grp_name)
        age = np.random.randint(30, 50, size=(time_points, 1))
        grp.create_dataset('age', data=age)
        grp.create_dataset('image', data=Case_image_list[i])
        grp.create_dataset('mask', data=Case_mask_list[i])


## create validation .hdf5 file 
with h5py.File('h5_Data/val_long.hdf5', 'w') as f:
    for i in range(len(Case_name_list)):
        if i < 4:
            pass
        else:
            time_points = len(Case_name_list[i])
            grp_name = 'subject_' + str(i-4).zfill(3)
            print(grp_name)
            grp = f.create_group(grp_name)
            age = np.random.randint(30, 50, size=(time_points, 1))
            grp.create_dataset('age', data=age)
            grp.create_dataset('image', data=Case_image_list[i])
            grp.create_dataset('mask', data=Case_mask_list[i])

subject_000
subject_001
subject_002
subject_003
subject_000


In [7]:
### This is for fine-tuning dataset
finetune_path = 'example_dataset/train_v4/'

## check if the file already exists
if not os.path.exists(finetune_path):
    os.makedirs(finetune_path)
    
finetune_path_dataset = os.path.join(finetune_path, 'train_image_seg_3d.hdf5')
with h5py.File(finetune_path_dataset, 'w') as f:
    for i in range(1):
        time_points = len(Case_name_list[i])
        grp_name = 'img_seg_pair'
        grp = f.create_group(grp_name)
        age = np.random.randint(30, 50, size=(time_points, 1))
        grp.create_dataset('age', data=age)
        grp.create_dataset('t2', data=Case_image_list[i])

        ### one-hot encoding mask
        mask = Case_mask_list[i].astype(int)
        one_hot_mask = np.eye(2)[mask]
        print(f"one hot mask shape: {one_hot_mask.shape}")
        one_hot_mask = np.moveaxis(one_hot_mask, -1, 0).astype(np.float32)
        print(f"one hot mask shape: {one_hot_mask.shape}")
        grp.create_dataset('seg', data=one_hot_mask)

one hot mask shape: (4, 181, 217, 181, 2)
one hot mask shape: (2, 4, 181, 217, 181)
